In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Formatting settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print("Setup complete!")

In [ ]:
# Load dataset
df = pd.read_csv('used_cars.csv')  # or path to your dataset file

print("--- Dataset Info ---")
df.info()

print("\n--- Summary Statistics (Numerical) ---")
display(df.describe())

print("\n--- Summary Statistics (Categorical) ---")
display(df.describe(include=['O']))

In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

# Check missing and duplicate values
print("\nMissing Values Count:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print(f"\nTotal Duplicate Records: {df.duplicated().sum()}")

# Unique values in categorical features
for col in cat_cols[:5]:  # view first few categorical features
    print(f"\nUnique values in {col}: {df[col].nunique()}")

In [ ]:
# Distribution plot for target variable (e.g., price)
if 'price' in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df['price'], kde=True)
    plt.title("Price Distribution")
    plt.show()

# Boxplot to inspect potential outliers
if 'milage' in df.columns or 'mileage' in df.columns:
    col_name = 'milage' if 'milage' in df.columns else 'mileage'
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col_name])
    plt.title(f"Outliers in {col_name}")
    plt.show()

In [ ]:
# 1. Remove duplicate records
df = df.drop_duplicates().reset_index(drop=True)

# 2. Convert text/numeric columns to clean numbers (e.g., removing '$', ',', 'mi', 'hp')
def clean_numeric(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace('$', '').replace(',', '').replace('mi.', '').replace('HP', '').strip()
    try:
        return float(val_str)
    except ValueError:
        return np.nan

# Clean common numeric columns if present as strings
for col in ['price', 'milage', 'mileage', 'hp']:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = df[col].apply(clean_numeric)

# 3. Handle missing values (Imputation)
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=['object']).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum().sum())

In [ ]:
current_year = datetime.now().year

# Feature 1: Car Age
if 'model_year' in df.columns:
    df['car_age'] = current_year - df['model_year']
elif 'year' in df.columns:
    df['car_age'] = current_year - df['year']

# Feature 2: Mileage Per Year
mileage_col = 'milage' if 'milage' in df.columns else ('mileage' if 'mileage' in df.columns else None)
if mileage_col and 'car_age' in df.columns:
    df['mileage_per_year'] = df[mileage_col] / (df['car_age'].replace(0, 1))

# Feature 3: Brand Category / Luxury Classification
luxury_brands = ['BMW', 'Mercedes-Benz', 'Audi', 'Porsche', 'Lexus', 'Jaguar', 'Land Rover', 'Tesla']
if 'brand' in df.columns:
    df['is_luxury_brand'] = df['brand'].apply(lambda x: 1 if str(x) in luxury_brands else 0)

# Feature 4: Accident History Indicator
if 'accident' in df.columns:
    df['has_accident_reported'] = df['accident'].apply(lambda x: 1 if 'accident' in str(x).lower() and 'no' not in str(x).lower() else 0)

# Feature 5: Clean Title Indicator
if 'clean_title' in df.columns:
    df['is_clean_title'] = df['clean_title'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

print("Engineered Features Sample:")
display(df.head())

In [ ]:
# Save transformed data
df.to_csv('cleaned_used_cars.csv', index=False)
print("Saved cleaned dataset successfully to cleaned_used_cars.csv")